# Combine simulation inputs and results

Combines 3 batches of simulation inputs/results + the MPM results file into a single long-format CSV with one row per simulation.

**Layout:**
- `Input_sweep` / `Results_sweep`: 200 sims
- `Input_Theave` / `Results_Theave`: 120 sims
- `Input_Tpitch` / `Results_Tpitch`: 120 sims
- Total: 440 sims
- `Results_all_MPM` and `Input_all` follow the same order: sweep → Theave → Tpitch

**Per-batch results:** 9 metrics per simulation (std/max/mean for Z, pitch, roll)

**MPM results:** 3 metrics per simulation (MPM Z, MPM pitch, MPM roll)

## 1. Imports and configuration

In [40]:
import pandas as pd
from pathlib import Path

UPLOADS = Path('C:\\Users\\verav\\Desktop\\Studie\\Afstuderen\\PHASE2_v2\\PHASE2_V2\\Simulaties_full\\Merge results\\Uploads')
OUTPUT = Path('C:\\Users\\verav\\Desktop\\Studie\\Afstuderen\\PHASE2_v2\\PHASE2_V2\\Simulaties_full\\allresults.csv')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

BATCHES = [
    ('sweep',  'Input_sweep.xlsx',  'Results_sweep.xlsx',  200),
    ('Theave', 'Input_Theave.xlsx', 'Results_Theave.xlsx', 120),
    ('Tpitch', 'Input_Tpitch.xlsx', 'Results_Tpitch.xlsx', 120),
]

RESULT_METRICS = ['std Z', 'std pitch', 'std roll',
                  'max Z', 'max pitch', 'max roll',
                  'mean Z', 'mean pitch', 'mean roll']
N_RESULT = len(RESULT_METRICS)  # 9

MPM_METRICS = ['MPM Z', 'MPM pitch', 'MPM roll']
N_MPM = len(MPM_METRICS)  # 3

## 2. Loader functions

In [41]:
def load_input(path: Path, n_sims_expected: int) -> pd.DataFrame:
    """Input layout: row 0 = title, row 1 = headers (with %), rows 2.. = data."""
    raw = pd.read_excel(path, sheet_name='Blad1', header=None)
    headers = [str(h).lstrip('%') for h in raw.iloc[1].tolist()]
    df = raw.iloc[2:].copy()
    df.columns = headers
    df = df.reset_index(drop=True)
    if len(df) != n_sims_expected:
        print(f'  WARNING: {path.name} has {len(df)} sims, expected {n_sims_expected}')
    return df

In [42]:
def load_results(path: Path, n_sims_expected: int) -> pd.DataFrame:
    """Results layout: row 0 = repeating labels, row 1 = values.
    9 columns per simulation, in metric order. Trailing extra cols are dropped."""
    raw = pd.read_excel(path, sheet_name='Blad1', header=None)
    expected_cols = n_sims_expected * N_RESULT

    if raw.shape[1] < expected_cols:
        print(f'  WARNING: {path.name} has {raw.shape[1]} cols, '
              f'expected {expected_cols} — cannot build {n_sims_expected} sims')
    elif raw.shape[1] > expected_cols:
        extra = raw.shape[1] - expected_cols
        extra_labels = raw.iloc[0, expected_cols:].tolist()
        print(f'  WARNING: {path.name} has {extra} extra trailing cols '
              f'({extra_labels}) — dropping them')
        raw = raw.iloc[:, :expected_cols]

    # Verify the metric pattern on the first block (case-insensitive)
    first_block_labels = [str(x).strip() for x in raw.iloc[0, :N_RESULT].tolist()]
    if [s.lower() for s in first_block_labels] != [s.lower() for s in RESULT_METRICS]:
        print(f'  WARNING: {path.name} first-block labels {first_block_labels} '
              f'do not match expected {RESULT_METRICS}')

    values = raw.iloc[1].to_numpy().reshape(n_sims_expected, N_RESULT)
    return pd.DataFrame(values, columns=RESULT_METRICS)

In [43]:
def load_mpm(path: Path, n_sims_total: int) -> pd.DataFrame:
    """MPM layout: row 0 = good labels, row 1 = ignore, row 2 = values.
    3 columns per simulation."""
    raw = pd.read_excel(path, sheet_name='Blad1', header=None)
    expected_cols = n_sims_total * N_MPM
    if raw.shape[1] != expected_cols:
        print(f'  WARNING: {path.name} has {raw.shape[1]} cols, '
              f'expected {expected_cols} ({n_sims_total} sims x {N_MPM})')

    first_block_labels = raw.iloc[0, :N_MPM].tolist()
    if first_block_labels != MPM_METRICS:
        print(f'  WARNING: {path.name} first-block labels {first_block_labels} '
              f'do not match expected {MPM_METRICS}')

    values = raw.iloc[2].to_numpy().reshape(n_sims_total, N_MPM)
    return pd.DataFrame(values, columns=MPM_METRICS)

## 3. Load per-batch inputs and results

In [44]:
batch_frames = []
for name, in_file, res_file, n in BATCHES:
    print(f'Loading batch: {name}')
    inp = load_input(UPLOADS / in_file, n)
    res = load_results(UPLOADS / res_file, n)

    if name == 'sweep':
           inp['Tp'] = pd.to_numeric(inp['Tp'], errors='coerce') * 1.09
           print(f'  Applied Tp x 1.09 correction to sweep batch')
    combined = pd.concat([inp.reset_index(drop=True),
                          res.reset_index(drop=True)], axis=1)
    batch_frames.append(combined)

all_batches = pd.concat(batch_frames, ignore_index=True)
print(f'\nTotal sims from batches: {len(all_batches)}')
all_batches.head()

Loading batch: sweep
  Applied Tp x 1.09 correction to sweep batch
Loading batch: Theave
Loading batch: Tpitch

Total sims from batches: 440


,filename,Hs,Tp,Tz,Direction,Damp_Heave,Damp_Roll,Damp_Pitch,Lin_Heave,Quad_Heave,...,Quad_Pitch,std Z,std pitch,std roll,max Z,max pitch,max roll,mean Z,mean pitch,mean roll
0,ymlfiles\Case_001_H2_T4_dir0_low.yml,2,4.36,3.1,0,low,low,low,1,1,...,200,0.128777,1.726431,0.000117,-0.510098,6.252082,-0.000981,-0.941013,0.305212,-0.002035
1,ymlfiles\Case_002_H2_T4_dir45_low.yml,2,4.36,3.1,45,low,low,low,1,1,...,200,0.139251,0.779213,1.203765,-0.487374,3.088617,4.698351,-0.940477,0.318843,0.004457
2,ymlfiles\Case_003_H2_T4_dir90_low.yml,2,4.36,3.1,90,low,low,low,1,1,...,200,0.123696,0.761861,0.726094,-0.553037,3.148292,2.801176,-0.940655,0.297572,-0.00676
3,ymlfiles\Case_004_H2_T4_dir135_low.yml,2,4.36,3.1,135,low,low,low,1,1,...,200,0.135579,0.589055,0.402547,-0.454809,2.657852,1.696876,-0.93833,0.309997,0.001436
4,ymlfiles\Case_005_H2_T4_dir180_low.yml,2,4.36,3.1,180,low,low,low,1,1,...,200,0.158334,1.895306,0.000134,-0.414127,7.459907,-0.000921,-0.941492,0.310223,-0.002033


## 4. Load Input_all and MPM results

In [45]:
input_all = load_input(UPLOADS / 'Input_all.xlsx', 440)
mpm = load_mpm(UPLOADS / 'Results_all_MPM.xlsx', 440)
mpm.head()

,MPM Z,MPM pitch,MPM roll
0,-0.438006,7.157576,-0.001576
1,-0.397687,3.406709,4.760784
2,-0.459716,3.291664,2.885859
3,-0.409455,2.657158,1.603464
4,-0.322681,7.820711,-0.001509


## 5. Verify filename order

The MPM results are positionally linked to `Input_all`. We assume `Input_all` is in the order `sweep → Theave → Tpitch`. Verify this against the concatenated batch filenames; warn on any mismatch.

In [46]:
fn_all = input_all['filename'].astype(str).tolist()
fn_concat = all_batches['filename'].astype(str).tolist()

if len(fn_all) != len(fn_concat):
    print(f'  WARNING: length mismatch — Input_all={len(fn_all)}, '
          f'concatenated batches={len(fn_concat)}')
else:
    mismatches = [(i, a, b) for i, (a, b) in enumerate(zip(fn_all, fn_concat)) if a != b]
    if mismatches:
        print(f'  WARNING: {len(mismatches)} filename mismatches. First few:')
        for i, a, b in mismatches[:5]:
            print(f'    row {i}: Input_all={a!r} vs batches={b!r}')
    else:
        print('  OK — filenames match in order.')

  OK — filenames match in order.


## 6. Combine and save

In [47]:
final = pd.concat([all_batches.reset_index(drop=True),
                   mpm.reset_index(drop=True)], axis=1)

final.to_csv(OUTPUT, index=False)
print(f'Written {len(final)} rows x {final.shape[1]} cols to {OUTPUT}')
final.head()

Written 440 rows x 26 cols to C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_v2\PHASE2_V2\Simulaties_full\allresults.csv


,filename,Hs,Tp,Tz,Direction,Damp_Heave,Damp_Roll,Damp_Pitch,Lin_Heave,Quad_Heave,...,std roll,max Z,max pitch,max roll,mean Z,mean pitch,mean roll,MPM Z,MPM pitch,MPM roll
0,ymlfiles\Case_001_H2_T4_dir0_low.yml,2,4.36,3.1,0,low,low,low,1,1,...,0.000117,-0.510098,6.252082,-0.000981,-0.941013,0.305212,-0.002035,-0.438006,7.157576,-0.001576
1,ymlfiles\Case_002_H2_T4_dir45_low.yml,2,4.36,3.1,45,low,low,low,1,1,...,1.203765,-0.487374,3.088617,4.698351,-0.940477,0.318843,0.004457,-0.397687,3.406709,4.760784
2,ymlfiles\Case_003_H2_T4_dir90_low.yml,2,4.36,3.1,90,low,low,low,1,1,...,0.726094,-0.553037,3.148292,2.801176,-0.940655,0.297572,-0.00676,-0.459716,3.291664,2.885859
3,ymlfiles\Case_004_H2_T4_dir135_low.yml,2,4.36,3.1,135,low,low,low,1,1,...,0.402547,-0.454809,2.657852,1.696876,-0.93833,0.309997,0.001436,-0.409455,2.657158,1.603464
4,ymlfiles\Case_005_H2_T4_dir180_low.yml,2,4.36,3.1,180,low,low,low,1,1,...,0.000134,-0.414127,7.459907,-0.000921,-0.941492,0.310223,-0.002033,-0.322681,7.820711,-0.001509


In [48]:
final.describe()

,filename,Hs,Tp,Tz,Direction,Damp_Heave,Damp_Roll,Damp_Pitch,Lin_Heave,Quad_Heave,...,std roll,max Z,max pitch,max roll,mean Z,mean pitch,mean roll,MPM Z,MPM pitch,MPM roll
count,440,440,440.00,440.0,440,440,440,440,440,440,...,440.000000,440.000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000
unique,440,3,11.00,11.0,5,4,4,4,3,3,...,440.000000,440.000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000
top,ymlfiles\Case_440_H4_T759_dir180_DOFbest.yml,2,4.36,3.1,0,low,low,low,1,1,...,0.000574,1.106,23.586004,0.000176,-0.920122,0.567869,-0.002033,1.794289,23.245404,0.000166
freq,1,220,40.00,40.0,88,110,110,110,220,220,...,1.000000,1.000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
